# MoSe2 Monolayer — NSOM Hyperspectral PL Mapping Analysis

**Author:** YoungBum Kim | PhD in Energy Science (Optics/Nanoscience)  
**Institution:** Sungkyunkwan University, Energy Science Department  
**Data:** Personal PhD measurement data — WITec Alpha 300 NSOM system (2018)  
**Instrument:** Near-field Scanning Optical Microscopy (NSOM) + Confocal PL  
**Sample:** MoSe2 monolayer on SiO2/Si substrate  
**Excitation:** 532 nm laser, 300 g/mm grating, center wavelength 750 nm  

---

## What this notebook does

1. Loads raw NSOM hyperspectral PL mapping data (60×60 pixels, 1024 pts/spectrum)
2. Removes cosmic ray artifacts automatically
3. Extracts peak position, intensity, and FWHM pixel-by-pixel
4. Generates 2D spatial maps revealing nanoscale optical inhomogeneity

## Key Results (10×10 pixel subset, 5×5 μm²)

| Parameter | Value |
|-----------|-------|
| A exciton peak position | ~756 meV (~1641 nm) |
| Peak position variation | 720 ~ 840 meV (120 meV range) |
| FWHM variation | 5 ~ 45 meV |
| Spatial resolution | ~100 nm (NSOM tip-limited) |

## Why this matters

> Conventional confocal PL averages over ~1 μm² diffraction-limited area.  
> NSOM resolves PL heterogeneity at **sub-100 nm scale** — invisible to standard microscopy.  
> Peak position shifts reveal local strain, defect density, and doping variations at the nanoscale.

---

## 1. Library Import

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter, find_peaks

print('Libraries loaded successfully!')
print(f'NumPy version: {np.__version__}')

## 2. Data Loading

Raw NSOM hyperspectral data exported from WITec Project FIVE:
- **X-Axis file**: Energy axis in meV (1024 points, 667.2 ~ 950.1 meV)
- **Y-Axis file**: CCD counts per pixel (60×60 spatial × 1024 spectral = 3,686,400 lines)

**Reading strategy**: Sequential `readline()` to avoid memory overflow (file size: ~140 MB)  
Subset selected: rows 40~50, cols 25~35 (10×10 pixels = central-lower region)

In [ ]:
# Configuration
MAP_X = 60
SPEC_POINTS = 1024
row_start, row_end = 40, 50   # 10 rows
col_start, col_end = 25, 35   # 10 columns

# Load X-axis (energy in meV)
x_axis = np.loadtxt('1 Export File (X-Axis).txt')
print(f'X-axis: {len(x_axis)} points')
print(f'Energy range: {x_axis.min():.1f} ~ {x_axis.max():.1f} meV')

# Load Y-axis data (sequential read to save memory)
print('\nLoading spectral data...')
spectra = []
with open('1 Export File (Y-Axis).txt', 'r') as f:
    for row in range(row_end):
        for col in range(MAP_X):
            spectrum = []
            for k in range(SPEC_POINTS):
                val = float(f.readline())
                spectrum.append(val)
            if row >= row_start and col_start <= col < col_end:
                spectra.append(spectrum)

spectra = np.array(spectra)
print(f'Loaded spectra: {len(spectra)} pixels')
print(f'Map shape: {int(len(spectra)**0.5)} x {int(len(spectra)**0.5)}')
print(f'Intensity range: {spectra.min():.1f} ~ {spectra.max():.1f} CCD cts')

## 3. Single Spectrum Analysis

Before mapping, inspect a representative single-pixel spectrum:
- **Cosmic ray removal**: Median-based spike detection and interpolation
- **Baseline subtraction**: 10th percentile as dark current estimate
- **Peak detection**: Savitzky-Golay smoothing + `find_peaks`

In [ ]:
def remove_cosmic_ray(spectrum, threshold=5):
    """Remove cosmic ray spikes by median-based outlier detection.
    
    Args:
        spectrum: 1D array of CCD counts
        threshold: spike detection threshold in units of std dev
    Returns:
        Cleaned spectrum with spikes replaced by linear interpolation
    """
    med = np.median(spectrum)
    std = np.std(spectrum)
    mask = spectrum > med + threshold * std
    s = spectrum.copy()
    for i in np.where(mask)[0]:
        if 0 < i < len(s) - 1:
            s[i] = (s[i-1] + s[i+1]) / 2
    return s

# Baseline subtraction
baseline = np.percentile(spectra, 10, axis=1, keepdims=True)
spectra_sub = spectra - baseline

# Single pixel analysis
spec0 = spectra_sub[0]
spec0_clean = remove_cosmic_ray(spec0)
spec0_smooth = savgol_filter(spec0_clean, 15, 3)

# Peak detection
peaks, _ = find_peaks(spec0_smooth,
                       height=spec0_smooth.max() * 0.1,
                       prominence=5,
                       distance=10)

print('=== Single Pixel Spectrum Analysis ===')
print(f'Detected peaks: {len(peaks)}')
for i, p in enumerate(peaks):
    print(f'  Peak {i+1}: {x_axis[p]:.1f} meV, intensity: {spec0_smooth[p]:.1f} CCD cts')

# Visualization
plt.figure(figsize=(10, 6))
plt.plot(x_axis, spec0, alpha=0.3, color='blue', label='Raw')
plt.plot(x_axis, spec0_clean, alpha=0.5, color='green', label='Cosmic ray removed')
plt.plot(x_axis, spec0_smooth, color='red', linewidth=2, label='Filtered')

for p in peaks:
    plt.axvline(x=x_axis[p], color='orange', linestyle='--', alpha=0.7)
    plt.annotate(f'{x_axis[p]:.0f} meV',
                 xy=(x_axis[p], spec0_smooth[p]),
                 xytext=(5, 5), textcoords='offset points', fontsize=9)

plt.xlabel('Energy (meV)', fontsize=12)
plt.ylabel('Intensity (CCD cts, baseline subtracted)', fontsize=12)
plt.title('MoSe2 NSOM — Single Pixel PL Spectrum with Cosmic Ray Removal', fontsize=13)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('MoSe2_single_spectrum.png', dpi=150)
plt.show()
print('Figure saved: MoSe2_single_spectrum.png')

## 4. Hyperspectral Map Analysis

Pixel-by-pixel peak extraction across the 10×10 spatial map:
- **Peak Intensity Map**: Spatial distribution of PL emission strength
- **Peak Position Map**: Local energy shifts due to strain/defects/doping
- **FWHM Map**: Linewidth broadening from crystal disorder

Each pixel undergoes: cosmic ray removal → baseline subtraction → Savitzky-Golay filtering → peak detection

In [ ]:
# Initialize maps
peak_positions  = np.zeros((10, 10))
peak_intensities = np.zeros((10, 10))
fwhm_map        = np.zeros((10, 10))

# Pixel-by-pixel extraction
for idx in range(100):
    row = idx // 10
    col = idx % 10

    spec       = spectra_sub[idx]
    spec_clean = remove_cosmic_ray(spec)
    spec_smooth = savgol_filter(spec_clean, 15, 3)

    peaks, _ = find_peaks(spec_smooth,
                           height=spec_smooth.max() * 0.3,
                           prominence=5)

    if len(peaks) > 0:
        main_peak = peaks[np.argmax(spec_smooth[peaks])]
        peak_positions[row, col]   = x_axis[main_peak]
        peak_intensities[row, col] = spec_smooth[main_peak]

        # FWHM calculation
        peak_val  = spec_smooth[main_peak]
        half_max  = peak_val / 2
        left_side  = spec_smooth[:main_peak]
        right_side = spec_smooth[main_peak:]
        left_idx   = (np.abs(left_side  - half_max)).argmin()
        right_idx  = (np.abs(right_side - half_max)).argmin()
        fwhm_map[row, col] = abs(x_axis[main_peak + right_idx] - x_axis[left_idx])

print('Peak extraction complete!')
print(f'Peak position range : {peak_positions[peak_positions>0].min():.1f} ~ {peak_positions.max():.1f} meV')
print(f'Peak intensity range: {peak_intensities[peak_intensities>0].min():.1f} ~ {peak_intensities.max():.1f} CCD cts')
print(f'FWHM range          : {fwhm_map[fwhm_map>0].min():.1f} ~ {fwhm_map.max():.1f} meV')

## 5. 2D Spatial Maps — Main Result

The three maps below reveal nanoscale optical heterogeneity in the MoSe2 monolayer:

- **Intensity map**: PL quenching regions indicate defect-rich areas
- **Position map**: Energy shifts (up to 120 meV) reflect local strain and defect-bound excitons  
- **FWHM map**: Linewidth broadening correlates with crystal disorder

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Peak Intensity Map
im1 = axes[0].imshow(peak_intensities, cmap='hot', origin='upper')
axes[0].set_title('Peak Intensity Map', fontsize=13, fontweight='bold')
axes[0].set_xlabel('X pixel (100 nm/pixel)', fontsize=11)
axes[0].set_ylabel('Y pixel (100 nm/pixel)', fontsize=11)
plt.colorbar(im1, ax=axes[0], label='Intensity (CCD cts)')

# Peak Position Map
im2 = axes[1].imshow(peak_positions, cmap='RdYlBu', origin='upper')
axes[1].set_title('Peak Position Map (meV)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('X pixel (100 nm/pixel)', fontsize=11)
axes[1].set_ylabel('Y pixel (100 nm/pixel)', fontsize=11)
plt.colorbar(im2, ax=axes[1], label='Energy (meV)')

# FWHM Map
im3 = axes[2].imshow(fwhm_map, cmap='viridis', origin='upper')
axes[2].set_title('FWHM Map (meV)', fontsize=13, fontweight='bold')
axes[2].set_xlabel('X pixel (100 nm/pixel)', fontsize=11)
axes[2].set_ylabel('Y pixel (100 nm/pixel)', fontsize=11)
plt.colorbar(im3, ax=axes[2], label='FWHM (meV)')

plt.suptitle(
    'MoSe2 NSOM Hyperspectral Analysis — 10×10 pixel (1×1 μm²)\n'
    'Excitation: 532 nm | Integration: 2 s/pixel | Spatial res: ~100 nm',
    fontsize=12
)
plt.tight_layout()
plt.savefig('MoSe2_3maps.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: MoSe2_3maps.png')

## 6. Statistical Summary

Quantitative summary of the spatial heterogeneity observed in the 10×10 pixel map.

In [ ]:
valid = peak_positions > 0

print('=' * 45)
print('  MoSe2 NSOM — Statistical Summary')
print('=' * 45)
print(f'  Analyzed pixels : {valid.sum()} / 100')
print()
print('  Peak Position (meV):')
print(f'    Mean  : {peak_positions[valid].mean():.1f}')
print(f'    Std   : {peak_positions[valid].std():.1f}')
print(f'    Range : {peak_positions[valid].min():.1f} ~ {peak_positions[valid].max():.1f}')
print()
print('  Peak Intensity (CCD cts):')
print(f'    Mean  : {peak_intensities[valid].mean():.1f}')
print(f'    Std   : {peak_intensities[valid].std():.1f}')
print(f'    Range : {peak_intensities[valid].min():.1f} ~ {peak_intensities[valid].max():.1f}')
print()
print('  FWHM (meV):')
print(f'    Mean  : {fwhm_map[valid].mean():.1f}')
print(f'    Std   : {fwhm_map[valid].std():.1f}')
print(f'    Range : {fwhm_map[valid].min():.1f} ~ {fwhm_map[valid].max():.1f}')
print('=' * 45)
print()
print('Physical interpretation:')
print('  - 120 meV position variation >> thermal broadening at RT (~26 meV)')
print('  - Suggests local strain, defect-bound excitons, or doping inhomogeneity')
print('  - NSOM resolution (~100 nm) reveals sub-diffraction heterogeneity')
print('    invisible to conventional confocal PL (~1 μm resolution)')

---

## Next Steps

- [ ] Expand analysis to 30×30 pixel region
- [ ] Apply scikit-learn clustering (K-means) to classify spectral regions automatically
- [ ] Compare with WSe2 bilayer NSOM data (CVD stacking vs. mechanical exfoliation)
- [ ] Lorentzian multi-peak fitting for defect vs. free exciton decomposition

---
*Data measured at Sungkyunkwan University, Energy Science Department (2018)*  
*Analysis code: github.com/kyb8801/semiconductor-ai-portfolio*